<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebooks/02_breakhis_all_magnification_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BreakHis All-Magnification Audit

This notebook extends the completed 40× BreakHis audit to the 100×, 200×, and 400× magnifications.

The purpose is to determine whether the additional magnifications can be used as natural acquisition-shift evaluation conditions while preserving the existing patient-level experimental design.

The existing 40× experiment remains the reference condition. The original patient-level split is not regenerated in this notebook.

This audit addresses the following questions:

1. Which patient IDs appear at 40×, 100×, 200×, and 400×?
2. Are the same patients represented consistently across magnifications?
3. Are diagnosis labels consistent for each patient across magnifications?
4. Are subtype labels consistent across magnifications?
5. Does the previously identified `14-13412` ductal/lobular anomaly appear at the other magnifications?
6. Are there additional duplicate or labeling anomalies outside the 40× subset?
7. Can the existing Development, Calibration, and Test patient assignments be reused for every magnification condition?

No model training or held-out performance evaluation is performed in this notebook.

In [1]:
# ============================================================
# Environment
# ============================================================

from pathlib import Path
import re
import hashlib

import numpy as np
import pandas as pd
from PIL import Image

SEED = 631

print("All-magnification BreakHis audit")
print("Seed:", SEED)

All-magnification BreakHis audit
Seed: 631


In [2]:
!pip install -q kagglehub

import kagglehub

breakhis_path = kagglehub.dataset_download(
    "ambarish/breakhis"
)

DATASET_ROOT = Path(breakhis_path)

print("Dataset root:")
print(DATASET_ROOT)

100%|██████████| 3.99G/3.99G [02:19<00:00, 30.7MB/s]

Extracting files...


Dataset root:
/root/.cache/kagglehub/datasets/ambarish/breakhis/versions/4


In [3]:
# ============================================================
# Find all original BreakHis PNG images
# ============================================================

all_pngs = sorted(
    DATASET_ROOT.rglob("*.png")
)

print("Total PNG files:", len(all_pngs))

assert len(all_pngs) == 7909

Total PNG files: 7909


In [4]:
# ============================================================
# Parse filename metadata
# ============================================================

pattern = re.compile(
    r"^SOB_([BM])_([A-Z]+)-(.+)-(40|100|200|400)-(\d+)\.png$"
)

diagnosis_map = {
    "B": "Benign",
    "M": "Malignant"
}

subtype_map = {
    "A": "Adenosis",
    "F": "Fibroadenoma",
    "TA": "Tubular adenoma",
    "PT": "Phyllodes tumor",
    "DC": "Ductal carcinoma",
    "LC": "Lobular carcinoma",
    "MC": "Mucinous carcinoma",
    "PC": "Papillary carcinoma"
}

rows = []

failed = []

for filepath in all_pngs:

    match = pattern.match(
        filepath.name
    )

    if match is None:
        failed.append(filepath.name)
        continue

    diagnosis_code = match.group(1)
    subtype_code = match.group(2)
    patient_id = match.group(3)
    magnification = match.group(4)
    image_number = int(match.group(5))

    rows.append({
        "filename": filepath.name,
        "filepath": str(filepath),
        "patient_id": patient_id,
        "diagnosis_code": diagnosis_code,
        "diagnosis": diagnosis_map[diagnosis_code],
        "subtype_code": subtype_code,
        "subtype": subtype_map[subtype_code],
        "magnification": magnification,
        "image_number": image_number
    })

metadata_all = pd.DataFrame(rows)

print("Parsed images:", len(metadata_all))
print("Failed filenames:", len(failed))

assert len(metadata_all) == 7909
assert len(failed) == 0

Parsed images: 7909
Failed filenames: 0


In [5]:
# ============================================================
# Magnification counts
# ============================================================

magnification_counts = (
    metadata_all[
        "magnification"
    ]
    .value_counts()
    .sort_index()
)

display(magnification_counts)

assert magnification_counts["40"] == 1995
assert magnification_counts["100"] == 2081
assert magnification_counts["200"] == 2013
assert magnification_counts["400"] == 1820

,count
magnification,
100,2081
200,2013
40,1995
400,1820


In [6]:
# ============================================================
# Unique patients by magnification
# ============================================================

patient_counts = (
    metadata_all
    .groupby("magnification")[
        "patient_id"
    ]
    .nunique()
)

display(patient_counts)

,patient_id
magnification,
100,81
200,81
40,81
400,81


In [7]:
patients_by_mag = {
    mag: set(
        metadata_all.loc[
            metadata_all["magnification"] == mag,
            "patient_id"
        ]
    )
    for mag in ["40", "100", "200", "400"]
}

for mag, patients in patients_by_mag.items():
    print(
        mag,
        "patients:",
        len(patients)
    )

40 patients: 81
100 patients: 81
200 patients: 81
400 patients: 81


In [8]:
# ============================================================
# Compare patient membership across magnifications
# ============================================================

reference_patients = patients_by_mag["40"]

for mag in ["100", "200", "400"]:

    missing_from_mag = (
        reference_patients -
        patients_by_mag[mag]
    )

    extra_in_mag = (
        patients_by_mag[mag] -
        reference_patients
    )

    print(f"\n40× vs {mag}×")

    print(
        "Missing relative to 40×:",
        len(missing_from_mag)
    )

    print(
        sorted(missing_from_mag)
    )

    print(
        "Extra relative to 40×:",
        len(extra_in_mag)
    )

    print(
        sorted(extra_in_mag)
    )


40× vs 100×
Missing relative to 40×: 0
[]
Extra relative to 40×: 0
[]

40× vs 200×
Missing relative to 40×: 0
[]
Extra relative to 40×: 0
[]

40× vs 400×
Missing relative to 40×: 0
[]
Extra relative to 40×: 0
[]


In [9]:
# ============================================================
# Patient diagnosis consistency
# ============================================================

patient_diagnosis_counts = (
    metadata_all
    .groupby("patient_id")[
        "diagnosis"
    ]
    .nunique()
)

diagnosis_conflicts = (
    patient_diagnosis_counts[
        patient_diagnosis_counts > 1
    ]
)

print(
    "Patients with conflicting "
    "binary diagnosis labels:",
    len(diagnosis_conflicts)
)

display(diagnosis_conflicts)

Patients with conflicting binary diagnosis labels: 0


,diagnosis
patient_id,


In [10]:
# ============================================================
# Patient subtype consistency
# ============================================================

patient_subtype_counts = (
    metadata_all
    .groupby("patient_id")[
        "subtype"
    ]
    .nunique()
)

subtype_conflicts = (
    patient_subtype_counts[
        patient_subtype_counts > 1
    ]
)

print(
    "Patients with multiple subtype labels:",
    len(subtype_conflicts)
)

display(subtype_conflicts)

Patients with multiple subtype labels: 1


,subtype
patient_id,
14-13412,2


In [11]:
# ============================================================
# Patient subtype consistency
# ============================================================

patient_subtype_counts = (
    metadata_all
    .groupby("patient_id")[
        "subtype"
    ]
    .nunique()
)

subtype_conflicts = (
    patient_subtype_counts[
        patient_subtype_counts > 1
    ]
)

print(
    "Patients with multiple subtype labels:",
    len(subtype_conflicts)
)

display(subtype_conflicts)

Patients with multiple subtype labels: 1


,subtype
patient_id,
14-13412,2


In [12]:
# ============================================================
# Known DC / LC anomaly
# ============================================================

anomaly_patient = (
    metadata_all[
        metadata_all[
            "patient_id"
        ] == "14-13412"
    ]
    .copy()
)

print(
    "Total files for 14-13412:",
    len(anomaly_patient)
)

display(
    anomaly_patient
    .groupby(
        [
            "magnification",
            "subtype"
        ]
    )
    .size()
    .rename("count")
    .reset_index()
)

Total files for 14-13412: 246


,magnification,subtype,count
0,100,Ductal carcinoma,33
1,100,Lobular carcinoma,33
2,200,Ductal carcinoma,32
3,200,Lobular carcinoma,32
4,40,Ductal carcinoma,32
5,40,Lobular carcinoma,32
6,400,Ductal carcinoma,26
7,400,Lobular carcinoma,26


In [13]:
# ============================================================
# Patient × magnification image counts
# ============================================================

patient_mag_counts = (
    metadata_all
    .groupby(
        [
            "patient_id",
            "magnification"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

display(
    patient_mag_counts.head(20)
)

print(
    "\nPatients missing at least "
    "one magnification:"
)

missing_any_mag = (
    patient_mag_counts == 0
).any(axis=1)

display(
    patient_mag_counts[
        missing_any_mag
    ]
)

magnification,100,200,40,400
patient_id,,,,
14-10147,12,22,15,15
14-10926,10,9,11,9
14-11031,17,14,14,15
14-11520,26,24,25,23
14-11951,25,28,27,20
14-12204,31,28,20,24
14-12312,41,35,32,38
14-12465,21,19,21,13
14-12773,25,21,26,18



Patients missing at least one magnification:


magnification,100,200,40,400
patient_id,,,,
